![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and LangChain to make a series of calls to a language model

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate Sequential Chain using langchain integration with watsonx models.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to chain `mistral-small-3-1-24b-instruct-2503` and `gpt-oss-120b` models to generate a sequence of creating a random question on a given topic and an answer to that question and also to make the user friends with LangChain framework, using simple chain (LLMChain) and the extended chain (SequentialChain) with the ChatWatsonx.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
3. [LangChain integration](#LangChain-integration)
4. [Sequential Chain experiment](#Sequential-Chain-experiment)
5. [AI Service](#AI-Service)
6. [Custom inference endpoint](#Custom-inference-endpoint)
7. [Scoring](#Scoring)
8. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "langchain>=0.3.25,<0.4" | tail -n 1
%pip install -U "langchain_ibm>=0.3.10,<0.4" | tail -n 1

### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=wx) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Go to **Manage** tab
- Copy `Space GUID` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below

In [3]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

Create an instance of APIClient with authentication details.

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials=credentials, space_id=space_id)

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

In [5]:
client.foundation_models.ChatModels.show()

{'GRANITE_4_H_SMALL': 'ibm/granite-4-h-small', 'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8': 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3-1-24b-instruct-2503', 'GPT_OSS_120B': 'openai/gpt-oss-120b'}


You need to specify `model_id`'s that will be used for inferencing:

In [6]:
model_id_1 = "mistralai/mistral-small-3-1-24b-instruct-2503"
model_id_2 = "openai/gpt-oss-120b"

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to documentation under `GenChatParamsMetaNames` class.

**Action:** If any complications please refer to the <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">documentation</a>.

In [7]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {
    GenParams.MAX_COMPLETION_TOKENS: 200,
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 1,
}

<a id="LangChain-integration"></a>
## LangChain integration

`ChatWatsonx` is a wrapper around watsonx.ai models that provide chain integration around the models.

**Action:** For more details about `CustomLLM` check the <a href="https://python.langchain.com/docs/modules/model_io/models/llms/custom_llm" target="_blank" rel="noopener no referrer">LangChain documentation</a>


### Initialize the `ChatWatsonx` class.

In [8]:
from langchain_ibm import ChatWatsonx

mistral_small_llm = ChatWatsonx(
    model_id=model_id_1,
    url=credentials.url,
    apikey=credentials.api_key,
    space_id=space_id,
    params=parameters,
)
mistral_medium_llm = ChatWatsonx(
    model_id=model_id_2,
    url=credentials.url,
    apikey=credentials.api_key,
    space_id=space_id,
)

You can print all set data about the ChatWatsonx object using the `dict()` method.

In [9]:
mistral_small_llm.dict()

{'_type': 'watsonx-chat'}

<a id="Sequential-Chain-experiment"></a>
## Sequential Chain experiment
The simplest type of sequential chain is called a `SequentialChain`, in which each step has a single input and output and the output of one step serves as the input for the following step.

The experiment will consist in generating a random question about any topic and answer the following question.

An object called `PromptTemplate` assists in generating prompts using a combination of user input, additional non-static data, and a fixed template string.

In our case we would like to create two `PromptTemplate` objects which will be responsible for creating a random question and answering it.

In [10]:
from langchain_core.prompts import PromptTemplate

prompt_1 = PromptTemplate(
    input_variables=["topic"],
    template="Describe a single specific person related to {topic}. The description must fit only this exact person. Do not use their name anywhere in your response. ",
)
prompt_2 = PromptTemplate(
    input_variables=["description"],
    template="{description} The previous text describes a person. Their name is: ",
)

In the chain below, `prompt_1` is passed to `mistral_small_llm`, then its response is added to `prompt_2`, which is then passed to `mistral_medium_llm`, after which we receive the response:

In [11]:
chain = prompt_1 | mistral_small_llm | prompt_2 | mistral_medium_llm

Generate random question and answer to topic.

In [12]:
from langchain.callbacks.tracers import ConsoleCallbackHandler

chain.invoke({"topic": "Formula 1"}, config={"callbacks": [ConsoleCallbackHandler()]})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "topic": "Formula 1"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "topic": "Formula 1"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatWatsonx] Entering LLM run with input:
{
  "prompts": [
    "Human: Describe a single specific person related to Formula 1. The description must fit only this exact person. Do not use their name anywhere in your response."
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatWatsonx] [1.79s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "This individual is known for their distinctive red overalls and has been a prominent figure in Formula 1 for over three decades. Born in Italy, they began their career in the sport as a mechanic before transitioning into a managerial role. They are renowned f

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: Who is described? The description: distinctive red overalls, prominent figure in F1 for over three decades, born in Italy, started as mechanic, became manager, strategic acumen, led most successful team, many championships, fiery personality, victory cigar. That\'s Team Principal of Scuderia Ferrari: Stefano Domenicali? He started as engineer, not mechanic. Red overalls reminiscent of Ferrari pit crew. The person is maybe "Maurizio Arrivabene"? He is Italian, but not mechanic. The unique celebrations with a victory cigar – that\'s "Franz Tost"? No. Actually the moustached guy with cigar is "Jean Todt"? He wasn\'t Italian. The red overalls and cigar reminds me of "Flavio Briatore"? He is Italian, but not mechanic, and not team boss of Ferrari. "Ruth"? No. Could be "Clemente"? Hmm.\n\nRed overalls also worn by "Team principal of Ferrari, Stefano Domenicali"? He used to wear red suit? He is also in F1 for de

<a id="AI-Service"></a>
## AI Service
Let's wrap the chain code within Python function that can be used to create an AI service.

### Function implementation.
Let's wrap the above chain code into function.

In [13]:
def chain_text_generator(context, url=credentials.url, parameters=parameters):
    from ibm_watsonx_ai import APIClient, Credentials
    from langchain_core.prompts import PromptTemplate
    from langchain_ibm import ChatWatsonx

    api_client = APIClient(
        credentials=Credentials(url=url, token=context.generate_token()),
        space_id=context.get_space_id(),
    )

    mistral_small_llm = ChatWatsonx(
        model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
        params=parameters,
        watsonx_client=api_client,
    )
    mistral_medium_llm = ChatWatsonx(
        model_id="openai/gpt-oss-120b",
        watsonx_client=api_client,
    )

    prompt_1 = PromptTemplate(
        input_variables=["topic"],
        template="Describe a single specific person related to {topic}. The description must fit only this exact person. Do not use their name anywhere in your response. ",
    )
    prompt_2 = PromptTemplate(
        input_variables=["description"],
        template="{description} The previous text describes a person. Their name is: ",
    )

    chain = prompt_1 | mistral_small_llm | prompt_2 | mistral_medium_llm

    def generate(context) -> dict:
        """Generates a description of a person based on provided topic and returns guesses on who that might be."""
        api_client.set_token(context.get_token())

        topic = context.get_json()["topic"]
        answer = chain.invoke({"topic": topic})

        return {"body": {"topic": topic, "answer": answer.content}}

    return generate

### Test the function
It is good practice to validate the code locally first.

In [14]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)

In [15]:
context.request_payload_json = {"topic": "Football"}

inference = chain_text_generator(context)
inference(context)

{'body': {'topic': 'Football',
  'answer': 'The description matches the Brazilian legend **Ronaldo\u202fLuís\u202fNazário\u202fde\u202fLima** – commonly known simply as **Ronaldo**.'}}

<a id="Custom-inference-endpoint"></a>
## Custom inference endpoint
Create the online deployment of python function.

### Store the function

In [16]:
sw_spec_id = client.software_specifications.get_id_by_name("genai-A25-py3.12")

meta_props = {
    client.repository.FunctionMetaNames.NAME: "SequenceChain LLM AI service",
    client.repository.FunctionMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}

ai_service_details = client.repository.store_ai_service(
    chain_text_generator, meta_props
)
ai_service_id = client.repository.get_ai_service_id(ai_service_details)

### Create online deployment

In [17]:
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "Deployment of LLMs chain AI service",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

ai_service_deployment = client.deployments.create(ai_service_id, meta_props=metadata)



######################################################################################

Synchronous deployment creation for id: '019feb12-9f1f-7671-9db2-ab28ae7a0e67' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='019feb13-24e4-702d-b8cb-cef1af31d3a4'
-----------------------------------------------------------------------------------------------




<a id="Scoring"></a>
## Scoring
Generate text using custom inference endpoint.

In [18]:
deployment_id = client.deployments.get_id(ai_service_deployment)

client.deployments.run_ai_service(deployment_id, {"topic": "Football"})

{'topic': 'Football',
 'answer': 'The description matches **Ronaldo Luís Nazário de Lima** – the Brazilian football legend commonly known simply as **Ronaldo**.'}

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use Sequential Chain using custom llm `ChatWatsonx`.
 
Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Lukasz Cmielowski (Former)**, PhD, Senior Technical Staff Member at IBM watsonx.ai

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.